In [ ]:

import os
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if not os.path.exists("datas") and os.path.exists("../datas"):
    os.chdir("..")

train_df = pd.read_csv("datas/train.csv")
test_df = pd.read_csv("datas/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'datas/train.csv'

In [ ]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

display(train_df[features + [target]].isnull().sum().to_frame('missing_values'))

fig = px.histogram(train_df, x='Sex', color='Survived', barmode='group',
                   title='Sobrevivência por sexo')
fig.show()

fig = px.histogram(train_df, x='Pclass', color='Survived', barmode='group',
                   title='Sobrevivência por classe')
fig.show()


,missing_values
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2
Survived,0


## Pré-processamento

In [ ]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

X = train_df[features].copy()
y = train_df[target].copy()
X_test_final = test_df[features].copy()

# Tratamento de valores ausentes
for col in ['Age', 'Fare']:
    median_value = X[col].median()
    X[col] = X[col].fillna(median_value)
    X_test_final[col] = X_test_final[col].fillna(median_value)

mode_embarked = X['Embarked'].mode()[0]
X['Embarked'] = X['Embarked'].fillna(mode_embarked)
X_test_final['Embarked'] = X_test_final['Embarked'].fillna(mode_embarked)

# Codificação categórica
X = pd.get_dummies(X, columns=['Sex', 'Embarked'], drop_first=True)
X_test_final = pd.get_dummies(X_test_final, columns=['Sex', 'Embarked'], drop_first=True)

# Garantir mesmas colunas entre treino e teste
X, X_test_final = X.align(X_test_final, join='left', axis=1, fill_value=0)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
X_train.head()


X_train: (712, 8)
X_valid: (179, 8)


,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
692,3,28.0,0,0,56.4958,True,False,True
481,2,28.0,0,0,0.0000,True,False,True
527,1,28.0,0,0,221.7792,True,False,True
855,3,18.0,0,1,9.3500,False,False,True
801,2,31.0,1,1,26.2500,False,False,True



## Treinamento do modelo

A Regressão Logística é um dos modelos mais usados para classificação binária.
Aqui ela aprende a probabilidade de sobrevivência com base nas variáveis.


In [ ]:

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

## Avaliação e geração da submissão

In [ ]:

valid_pred = model.predict(X_valid)
valid_acc = accuracy_score(y_valid, valid_pred)

print("Acurácia:", round(valid_acc, 4))
print("\nRelatório de classificação:")
print(classification_report(y_valid, valid_pred))

cm = confusion_matrix(y_valid, valid_pred)
cm_df = pd.DataFrame(cm, index=['Real_0', 'Real_1'], columns=['Pred_0', 'Pred_1'])
display(cm_df)

test_pred = model.predict(X_test_final)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('submission_logistic_regression.csv', index=False)
print("Arquivo salvo: submission_logistic_regression.csv")
submission.head()


Acurácia: 0.8045

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



,Pred_0,Pred_1
Real_0,98,12
Real_1,23,46


Arquivo salvo: submission_logistic_regression.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



## Interpretação rápida

Na Regressão Logística, os coeficientes ajudam a entender o efeito de cada variável.
Valores positivos tendem a aumentar a chance de sobrevivência.


In [ ]:

coef_df = pd.DataFrame({
    'Variavel': X_train.columns,
    'Coeficiente': model.coef_[0]
}).sort_values('Coeficiente', ascending=False)

coef_df


,Variavel,Coeficiente
6,Embarked_Q,0.280459
4,Fare,0.002235
1,Age,-0.038569
3,Parch,-0.071204
2,SibSp,-0.244477
7,Embarked_S,-0.382968
0,Pclass,-1.092587
5,Sex_male,-2.559549
